# Dataset Quality: `large_dataset_serv` vs `cosmic_large`

This notebook compares the training datasets used by:

- `diffusion_template/serv_new_runs/start_ba_large_ds_serv.sh`
- `diffusion_template/serv_new_runs/start_ba_cosm_new1.sh`

It focuses on factors that can make `cosmic_large` train worse despite being larger: target face size, target/ref construction, prompt quality, file validity, sampled image quality, and filtering tradeoffs.

## Training Setup Mirrored Here

`large_dataset_serv` uses `LargeDatasetTrain`, `batch_size=4`, `num_refs=1`, and `train_on_separate_image=true`, so each reference is a different full 1024 same-identity image.

`cosmic_large` uses `CosmicLargeTrain`, `batch_size=1`, and YAML default `num_refs=1`. Note: `start_ba_cosm_new1.sh` currently contains `datasets.train.cosmic_large_local.num_refs=1` while `train_dataset_name=cosmic_large`; that override targets the wrong config key, but the YAML default for `cosmic_large.num_refs` is already `1`.

In [ ]:
from pathlib import Path

# Server paths from all_datasets.yaml / launch scripts.
SERVER_LARGE_JSON = Path('/mnt/virtual_ai0001053-01309_SR006-nfs1/nasilaev/datasets/dataset_full/filtered_ids3_adj.json')
SERVER_LARGE_IMAGES = Path('/mnt/virtual_ai0001053-01309_SR006-nfs1/nasilaev/datasets/dataset_full/large_dataset_adj/large_dataset')

SERVER_COSMIC_JSON = Path('/mnt/virtual_ai0001053-01309_SR006-nfs1/bobkov/cosmic_data/gathered_data_cosmic_large_filtered.json')
SERVER_COSMIC_FACE_IMAGES = Path('/mnt/virtual_ai0001053-01309_SR006-nfs1/bobkov/cosmic_data/LAION-5B-Filtered-Large-Faces/laion1B-nolang')
SERVER_COSMIC_TARGET_ROOTS = [
    Path('/mnt/virtual_ai0001053-01309_SR006-nfs1/bobkov/cosmic_data/LAION-5B-Filtered-Large/laion1B-nolang'),
    Path('/mnt/virtual_ai0001053-01309_SR006-nfs1/bobkov/cosmic_data/LAION-5B-Filtered-Large'),
]

# Local fallbacks, useful when running this notebook from /home/kolyangg/rsrch/dataset_full.
LOCAL_DATASET_FULL = Path('/home/kolyangg/rsrch/dataset_full')
LOCAL_LARGE_JSON = LOCAL_DATASET_FULL / 'filtered_ids3_adj.json'
LOCAL_LARGE_IMAGES = LOCAL_DATASET_FULL / 'large_dataset_adj/large_dataset'
LOCAL_COSMIC_JSON = LOCAL_DATASET_FULL / 'cosmic_large/gathered_data_cosmic_large_filtered.json'
LOCAL_COSMIC_FACE_IMAGES = LOCAL_DATASET_FULL / 'cosmic_large/LAION-5B-Filtered-Large-Faces/laion1B-nolang'
LOCAL_COSMIC_TARGET_ROOTS = [
    LOCAL_DATASET_FULL / 'LAION-5B-Filtered-Large/laion1B-nolang',
    LOCAL_DATASET_FULL / 'LAION-5B-Filtered-Large',
]

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return paths[0]

LARGE_JSON = first_existing([SERVER_LARGE_JSON, LOCAL_LARGE_JSON])
LARGE_IMAGES = first_existing([SERVER_LARGE_IMAGES, LOCAL_LARGE_IMAGES])
COSMIC_JSON = first_existing([SERVER_COSMIC_JSON, LOCAL_COSMIC_JSON])
COSMIC_FACE_IMAGES = first_existing([SERVER_COSMIC_FACE_IMAGES, LOCAL_COSMIC_FACE_IMAGES])
COSMIC_TARGET_ROOTS = SERVER_COSMIC_TARGET_ROOTS + LOCAL_COSMIC_TARGET_ROOTS
COSMIC_TARGET_ROOT = first_existing(COSMIC_TARGET_ROOTS)
COSMIC_PATH_PREFIX_TO_STRIP = 'LAION-5B-Filtered-Large/laion1B-nolang'

# Controls.
RANDOM_SEED = 42
NUM_EXAMPLES = 12
IMAGE_STATS_SAMPLE = 1500  # lower if network FS is slow
CHECK_FILE_EXISTS_SAMPLE = 5000
COSMIC_NUM_REFS_USED_IN_TRAINING = 1
LARGE_TRAIN_ON_SEPARATE_IMAGE = True

print('LARGE_JSON:', LARGE_JSON, LARGE_JSON.exists())
print('LARGE_IMAGES:', LARGE_IMAGES, LARGE_IMAGES.exists())
print('COSMIC_JSON:', COSMIC_JSON, COSMIC_JSON.exists())
print('COSMIC_FACE_IMAGES:', COSMIC_FACE_IMAGES, COSMIC_FACE_IMAGES.exists())
print('COSMIC_TARGET_ROOT:', COSMIC_TARGET_ROOT, COSMIC_TARGET_ROOT.exists())

In [ ]:
import json
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps, ImageStat, UnidentifiedImageError
import matplotlib.pyplot as plt

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def bbox_metrics(box, image_size=1024):
    if box is None or len(box) != 4:
        return dict(valid_bbox=False)
    x0, y0, x1, y1 = [float(v) for v in box]
    w = x1 - x0
    h = y1 - y0
    area = max(0.0, w) * max(0.0, h)
    return {
        'valid_bbox': w > 0 and h > 0,
        'face_w': w,
        'face_h': h,
        'face_min_side': min(w, h),
        'face_area': area,
        'face_area_pct': area / float(image_size * image_size),
        'face_aspect': w / h if h > 0 else np.nan,
        'face_cx_pct': ((x0 + x1) / 2.0) / image_size,
        'face_cy_pct': ((y0 + y1) / 2.0) / image_size,
    }

def text_metrics(text):
    text = text if isinstance(text, str) else ''
    toks = text.split()
    return {
        'prompt_chars': len(text),
        'prompt_words': len(toks),
        'has_img_token': 'img' in toks,
    }

def describe_numeric(df, cols):
    rows = []
    for c in cols:
        s = pd.to_numeric(df[c], errors='coerce').dropna()
        if len(s) == 0:
            continue
        rows.append({
            'column': c,
            'n': len(s),
            'mean': s.mean(),
            'std': s.std(),
            'min': s.min(),
            'p05': s.quantile(0.05),
            'p25': s.quantile(0.25),
            'median': s.median(),
            'p75': s.quantile(0.75),
            'p90': s.quantile(0.90),
            'p95': s.quantile(0.95),
            'max': s.max(),
        })
    return pd.DataFrame(rows)

def safe_rel(path):
    return str(path).lstrip('/')

def strip_prefix(path, prefix):
    s = safe_rel(path)
    prefix = prefix.strip('/')
    if s.startswith(prefix + '/'):
        return s[len(prefix) + 1:]
    return s

In [ ]:
def flatten_large_dataset(data):
    rows = []
    identity_counts = {identity: len(records) for identity, records in data.items() if isinstance(records, dict)}
    for identity, records in data.items():
        if not isinstance(records, dict):
            continue
        for image_id, meta in records.items():
            rel_path = f'{identity}/{image_id}.jpg'
            body_crop = meta.get('body_crop')
            body_w = body_h = np.nan
            if isinstance(body_crop, list) and len(body_crop) == 4:
                # LargeDatasetTrain order: [x0, x1, y0, y1]
                body_w = body_crop[1] - body_crop[0]
                body_h = body_crop[3] - body_crop[2]
            prompt = meta.get('text', '') or 'img person'
            row = {
                'dataset': 'large_dataset_serv',
                'identity': identity,
                'image_id': image_id,
                'image_path': rel_path,
                'target_path': str(LARGE_IMAGES / rel_path),
                'prompt': prompt,
                'num_identity_images': identity_counts.get(identity, 0),
                'has_same_id_ref_candidate': identity_counts.get(identity, 0) > 1,
                'body_crop_w': body_w,
                'body_crop_h': body_h,
            }
            row.update(bbox_metrics(meta.get('new_face_crop')))
            row.update(text_metrics(prompt))
            rows.append(row)
    return pd.DataFrame(rows)

def cosmic_relative(path):
    s = safe_rel(path)
    s2 = strip_prefix(s, COSMIC_PATH_PREFIX_TO_STRIP)
    if s2 != s:
        return s2
    markers = [
        'LAION-5B-Filtered-Large/laion1B-nolang/',
        'LAION-5B-Filtered-Large-Faces/laion1B-nolang/',
        'laion1B-nolang/',
    ]
    for marker in markers:
        if marker in s:
            return s.split(marker, 1)[1]
    return s

def cosmic_target_path(path):
    s = safe_rel(path)
    rel = cosmic_relative(s)
    candidates = []
    for root in COSMIC_TARGET_ROOTS:
        candidates.extend([root / rel, root / s])
    # Candidate matching CosmicLargeTrain dataset_root / original path.
    if len(COSMIC_FACE_IMAGES.parents) >= 2:
        candidates.append(COSMIC_FACE_IMAGES.parents[1] / s)
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

def cosmic_target_path_fast(path):
    return COSMIC_TARGET_ROOT / cosmic_relative(path)

def cosmic_face_path(path):
    s = safe_rel(path)
    rel = cosmic_relative(s)
    candidates = [COSMIC_FACE_IMAGES / rel, COSMIC_FACE_IMAGES / s]
    if len(COSMIC_FACE_IMAGES.parents) >= 2:
        candidates.append(COSMIC_FACE_IMAGES.parents[1] / s)
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

def cosmic_face_path_fast(path):
    return COSMIC_FACE_IMAGES / cosmic_relative(path)

def flatten_cosmic_dataset(data, min_face_res=64, num_refs=1, only_complex_background=False):
    rows = []
    ref_rows = []
    for image_path, meta in data.items():
        bbox = meta.get('face_crop_new')
        bm = bbox_metrics(bbox)
        if not bm.get('valid_bbox', False):
            continue
        kept_by_class = True
        if bm['face_min_side'] < min_face_res:
            kept_by_class = False
        if only_complex_background and (meta.get('has_simple_back', False) or meta.get('is_simp', False)):
            kept_by_class = False
        face_paths = meta.get('face_paths') or []
        if len(face_paths) < num_refs:
            kept_by_class = False
        prompt = ', '.join([meta.get('facial_caption', ''), meta.get('pose_caption', ''), meta.get('background_caption', '')])
        body_crop = meta.get('body_crop')
        body_w = body_h = np.nan
        if isinstance(body_crop, list) and len(body_crop) == 4:
            # CosmicLargeTrain order: [x0, y0, x1, y1]
            body_w = body_crop[2] - body_crop[0]
            body_h = body_crop[3] - body_crop[1]
        row = {
            'dataset': 'cosmic_large',
            'image_path': image_path,
            'target_path': str(cosmic_target_path_fast(image_path)),
            'prompt': prompt,
            'num_face_paths': len(face_paths),
            'kept_by_current_class': kept_by_class,
            'is_simple_background': bool(meta.get('has_simple_back', False) or meta.get('is_simp', False)),
            'body_crop_w': body_w,
            'body_crop_h': body_h,
        }
        row.update(bm)
        row.update(text_metrics(prompt))
        rows.append(row)

        face_bboxes = meta.get('face_bboxes') or {}
        for face_path in face_paths:
            rb = face_bboxes.get(face_path) or face_bboxes.get(safe_rel(face_path)) or face_bboxes.get(cosmic_relative(face_path))
            rr = {'image_path': image_path, 'ref_path': face_path, 'ref_abs_path': str(cosmic_face_path_fast(face_path))}
            rr.update({f'ref_{k}': v for k, v in bbox_metrics(rb, image_size=256).items()})
            ref_rows.append(rr)
    return pd.DataFrame(rows), pd.DataFrame(ref_rows)

large_raw = load_json(LARGE_JSON)
cosmic_raw = load_json(COSMIC_JSON)
large_df = flatten_large_dataset(large_raw)
cosmic_df, cosmic_refs_df = flatten_cosmic_dataset(cosmic_raw, min_face_res=64, num_refs=COSMIC_NUM_REFS_USED_IN_TRAINING)

print('large rows:', len(large_df), 'identities:', large_df['identity'].nunique())
print('cosmic rows:', len(cosmic_df), 'kept by current class:', int(cosmic_df['kept_by_current_class'].sum()), 'ref rows:', len(cosmic_refs_df))
display(large_df.head(2))
display(cosmic_df.head(2))

In [ ]:
summary_cols = [
    'face_min_side', 'face_area_pct', 'face_aspect', 'face_cx_pct', 'face_cy_pct',
    'body_crop_w', 'body_crop_h', 'prompt_words', 'prompt_chars',
]
combined = pd.concat([large_df, cosmic_df], ignore_index=True, sort=False)
display(combined.groupby('dataset').agg(
    n=('dataset', 'size'),
    median_face_min_side=('face_min_side', 'median'),
    p25_face_min_side=('face_min_side', lambda s: s.quantile(0.25)),
    p10_face_min_side=('face_min_side', lambda s: s.quantile(0.10)),
    median_face_area_pct=('face_area_pct', 'median'),
    median_prompt_words=('prompt_words', 'median'),
    has_img_token_rate=('has_img_token', 'mean'),
))

print('Detailed numeric summaries')
display(describe_numeric(large_df.assign(dataset='large_dataset_serv'), summary_cols))
display(describe_numeric(cosmic_df[cosmic_df.kept_by_current_class].assign(dataset='cosmic_large'), summary_cols))

print('Large same-id reference candidate rate:', large_df['has_same_id_ref_candidate'].mean())
print('Cosmic reference path count summary')
display(cosmic_df['num_face_paths'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.9, 0.95]))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plot_specs = [
    ('face_min_side', 'Target face min side, px'),
    ('face_area_pct', 'Target face area fraction'),
    ('prompt_words', 'Prompt words'),
    ('face_cx_pct', 'Face center X'),
    ('face_cy_pct', 'Face center Y'),
    ('body_crop_h', 'Body crop height'),
]
for ax, (col, title) in zip(axes.ravel(), plot_specs):
    for name, df, color in [('large', large_df, 'tab:blue'), ('cosmic', cosmic_df[cosmic_df.kept_by_current_class], 'tab:orange')]:
        vals = pd.to_numeric(df[col], errors='coerce').dropna()
        if col == 'face_area_pct':
            vals = vals.clip(0, 0.25)
        ax.hist(vals, bins=60, alpha=0.45, density=True, label=name, color=color)
    ax.set_title(title)
    ax.legend()
plt.tight_layout()

In [ ]:
def open_rgb(path):
    return Image.open(path).convert('RGB')

def load_large_target(row):
    img = open_rgb(Path(row['target_path']))
    return img

def load_cosmic_target(row, meta):
    path = Path(row['target_path'])
    if not path.exists():
        path = cosmic_target_path(row['image_path'])
    img = open_rgb(path)
    if img.size != (1024, 1024):
        body_crop = meta.get('body_crop')
        if isinstance(body_crop, list) and len(body_crop) == 4:
            img = img.crop((body_crop[0], body_crop[1], body_crop[2], body_crop[3]))
    return img

def clip_bbox_to_image(bbox, size):
    w, h = size
    x0, y0, x1, y1 = [float(v) for v in bbox]
    out = [max(0, min(w, x0)), max(0, min(h, y0)), max(0, min(w, x1)), max(0, min(h, y1))]
    if out[2] <= out[0] or out[3] <= out[1]:
        return None
    return out

def get_bigger_crop_with_bbox(img, face_bbox, scale=0.2):
    crop = [int(round(v)) for v in face_bbox]
    if crop[3] - crop[1] < crop[2] - crop[0]:
        diff = crop[2] - crop[0] - (crop[3] - crop[1])
        if diff % 2 != 0:
            crop[0] -= 1
            diff += 1
        crop[3] += diff // 2
        crop[1] -= diff // 2
    elif crop[2] - crop[0] < crop[3] - crop[1]:
        diff = crop[3] - crop[1] - (crop[2] - crop[0])
        if diff % 2 != 0:
            crop[1] -= 1
            diff += 1
        crop[2] += diff // 2
        crop[0] -= diff // 2
    to_add = int((crop[3] - crop[1]) * scale)
    w, h = img.size
    crop = [max(0, crop[0] - to_add), max(0, crop[1] - to_add), min(w, crop[2] + to_add), min(h, crop[3] + to_add)]
    cropped = img.crop(tuple(crop))
    new_bbox = [face_bbox[0] - crop[0], face_bbox[1] - crop[1], face_bbox[2] - crop[0], face_bbox[3] - crop[1]]
    return cropped, clip_bbox_to_image(new_bbox, cropped.size)

def image_quality_metrics(img):
    small = img.resize((256, 256))
    arr = np.asarray(small).astype(np.float32) / 255.0
    gray = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
    gy, gx = np.gradient(gray)
    sharpness = float(np.mean(gx * gx + gy * gy))
    sat = np.asarray(small.convert('HSV'))[..., 1].astype(np.float32) / 255.0
    return {
        'img_w': img.width,
        'img_h': img.height,
        'brightness_mean': float(gray.mean()),
        'brightness_std': float(gray.std()),
        'sharpness_grad_mean': sharpness,
        'saturation_mean': float(sat.mean()),
        'dark_pixel_rate': float((gray < 0.05).mean()),
        'bright_pixel_rate': float((gray > 0.95).mean()),
    }

def sample_image_quality(df, dataset_name, n=1000):
    sample = df.sample(min(n, len(df)), random_state=RANDOM_SEED).copy()
    rows = []
    for _, row in sample.iterrows():
        try:
            if dataset_name == 'large_dataset_serv':
                img = load_large_target(row)
            else:
                img = load_cosmic_target(row, cosmic_raw[row['image_path']])
            q = image_quality_metrics(img)
            q.update({'dataset': dataset_name, 'image_path': row['image_path'], 'opened_ok': True})
        except Exception as e:
            q = {'dataset': dataset_name, 'image_path': row['image_path'], 'opened_ok': False, 'error': repr(e)}
        rows.append(q)
    return pd.DataFrame(rows)

large_quality = sample_image_quality(large_df, 'large_dataset_serv', IMAGE_STATS_SAMPLE)
cosmic_quality = sample_image_quality(cosmic_df[cosmic_df.kept_by_current_class], 'cosmic_large', IMAGE_STATS_SAMPLE)
quality = pd.concat([large_quality, cosmic_quality], ignore_index=True, sort=False)
display(quality.groupby('dataset').agg(open_rate=('opened_ok', 'mean'), n=('dataset', 'size')))
display(describe_numeric(quality[quality.opened_ok], ['img_w', 'img_h', 'brightness_mean', 'brightness_std', 'sharpness_grad_mean', 'saturation_mean', 'dark_pixel_rate', 'bright_pixel_rate']))

In [ ]:
def check_target_existence(df, n=5000):
    sample = df.sample(min(n, len(df)), random_state=RANDOM_SEED)
    rows = []
    for _, row in sample.iterrows():
        rows.append({'dataset': row['dataset'], 'image_path': row['image_path'], 'target_exists': Path(row['target_path']).exists()})
    return pd.DataFrame(rows)

existence = pd.concat([
    check_target_existence(large_df, CHECK_FILE_EXISTS_SAMPLE),
    check_target_existence(cosmic_df[cosmic_df.kept_by_current_class], CHECK_FILE_EXISTS_SAMPLE),
], ignore_index=True)
display(existence.groupby('dataset')['target_exists'].agg(['mean', 'sum', 'count']))

if len(cosmic_refs_df):
    ref_sample = cosmic_refs_df.sample(min(CHECK_FILE_EXISTS_SAMPLE, len(cosmic_refs_df)), random_state=RANDOM_SEED).copy()
    ref_sample['ref_exists'] = ref_sample['ref_abs_path'].map(lambda p: Path(p).exists())
    print('Cosmic sampled ref file existence')
    display(ref_sample['ref_exists'].agg(['mean', 'sum', 'count']))
    print('Cosmic ref bbox stats, image_size assumed 256 for raw face-crop metadata')
    display(describe_numeric(cosmic_refs_df, ['ref_face_min_side', 'ref_face_area_pct', 'ref_face_aspect']))

In [ ]:
def draw_bbox(img, bbox, color='red', width=4, label=None):
    out = img.copy()
    d = ImageDraw.Draw(out)
    if bbox is not None:
        box = [int(round(v)) for v in bbox]
        for off in range(width):
            d.rectangle([box[0]-off, box[1]-off, box[2]+off, box[3]+off], outline=color)
        if label:
            d.text((box[0] + 3, max(0, box[1] - 14)), label, fill=color)
    return out

def get_large_ref(row):
    identity = row['identity']
    candidates = large_df[(large_df.identity == identity) & (large_df.image_path != row.image_path)]
    if len(candidates) == 0:
        return None, None
    ref = candidates.sample(1, random_state=RANDOM_SEED).iloc[0]
    img = load_large_target(ref)
    bbox = [ref.face_cx_pct, ref.face_cy_pct]  # placeholder overwritten below
    meta = large_raw[ref.identity][ref.image_id]
    return img, meta.get('new_face_crop')

def get_cosmic_ref(row):
    meta = cosmic_raw[row['image_path']]
    face_paths = meta.get('face_paths') or []
    if not face_paths:
        return None, None
    face_path = random.choice(face_paths)
    face_bboxes = meta.get('face_bboxes') or {}
    rb = face_bboxes.get(face_path) or face_bboxes.get(safe_rel(face_path)) or face_bboxes.get(cosmic_relative(face_path))
    ref_img = open_rgb(cosmic_face_path(face_path))
    return get_bigger_crop_with_bbox(ref_img, rb)

def show_examples(df, dataset_name, n=8, order='random'):
    if order == 'small_faces':
        rows = df.sort_values('face_min_side').head(n)
    elif order == 'large_faces':
        rows = df.sort_values('face_min_side', ascending=False).head(n)
    else:
        rows = df.sample(min(n, len(df)), random_state=RANDOM_SEED)
    fig, axes = plt.subplots(len(rows), 2, figsize=(8, 4 * len(rows)))
    if len(rows) == 1:
        axes = np.array([axes])
    for ax_row, (_, row) in zip(axes, rows.iterrows()):
        try:
            if dataset_name == 'large_dataset_serv':
                img = load_large_target(row)
                meta = large_raw[row.identity][row.image_id]
                ref_img, ref_bbox = get_large_ref(row)
                bbox = meta.get('new_face_crop')
            else:
                meta = cosmic_raw[row['image_path']]
                img = load_cosmic_target(row, meta)
                ref_img, ref_bbox = get_cosmic_ref(row)
                bbox = meta.get('face_crop_new')
            ax_row[0].imshow(draw_bbox(img.resize((512, 512)), [v / 2 for v in bbox], 'red', width=2, label='target'))
            ax_row[0].set_title(f"{dataset_name}: face_min={row.face_min_side:.0f}, area={row.face_area_pct:.3f}")
            ax_row[0].axis('off')
            if ref_img is not None:
                scale = 512 / max(ref_img.size)
                ref_resized = ref_img.resize((max(1, int(ref_img.width * scale)), max(1, int(ref_img.height * scale))))
                rb = [v * scale for v in ref_bbox] if ref_bbox is not None else None
                ax_row[1].imshow(draw_bbox(ref_resized, rb, 'lime', width=2, label='ref'))
                ax_row[1].set_title('sampled training ref')
            else:
                ax_row[1].text(0.5, 0.5, 'no ref', ha='center')
            ax_row[1].axis('off')
        except Exception as e:
            for ax in ax_row:
                ax.text(0.5, 0.5, repr(e), ha='center', wrap=True)
                ax.axis('off')
    plt.tight_layout()

show_examples(large_df, 'large_dataset_serv', n=NUM_EXAMPLES, order='random')
show_examples(cosmic_df[cosmic_df.kept_by_current_class], 'cosmic_large', n=NUM_EXAMPLES, order='random')

In [ ]:
# Worst-case visual audit: tiny target faces are a common reason for unstable face generation.
show_examples(large_df, 'large_dataset_serv', n=8, order='small_faces')
show_examples(cosmic_df[cosmic_df.kept_by_current_class], 'cosmic_large', n=8, order='small_faces')

## Optional: Face-Embedding Target/Ref Consistency

If `insightface` is installed on the server, enable the cell below. It estimates whether the sampled cosmic reference actually matches the target face. Low cosine similarity or many detection failures indicate noisy target/ref pairing.

In [ ]:
RUN_INSIGHTFACE_CONSISTENCY = False
FACE_CONSISTENCY_SAMPLE = 300

if RUN_INSIGHTFACE_CONSISTENCY:
    from insightface.app import FaceAnalysis
    app = FaceAnalysis(name='buffalo_l')
    app.prepare(ctx_id=0, det_size=(640, 640))

    def face_embedding(img):
        arr = np.asarray(img.convert('RGB'))[:, :, ::-1]
        faces = app.get(arr)
        if not faces:
            return None
        faces = sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]), reverse=True)
        emb = faces[0].embedding.astype(np.float32)
        return emb / max(np.linalg.norm(emb), 1e-8)

    rows = []
    for _, row in cosmic_df[cosmic_df.kept_by_current_class].sample(min(FACE_CONSISTENCY_SAMPLE, len(cosmic_df)), random_state=RANDOM_SEED).iterrows():
        try:
            meta = cosmic_raw[row['image_path']]
            tgt = load_cosmic_target(row, meta)
            x0, y0, x1, y1 = [int(v) for v in meta['face_crop_new']]
            tgt_face = tgt.crop((x0, y0, x1, y1))
            ref, _ = get_cosmic_ref(row)
            e1 = face_embedding(tgt_face)
            e2 = face_embedding(ref)
            cos = np.nan if e1 is None or e2 is None else float(np.dot(e1, e2))
            rows.append({'image_path': row['image_path'], 'cosine': cos, 'ok': np.isfinite(cos)})
        except Exception as e:
            rows.append({'image_path': row['image_path'], 'cosine': np.nan, 'ok': False, 'error': repr(e)})
    consistency = pd.DataFrame(rows)
    display(consistency['ok'].agg(['mean', 'sum', 'count']))
    display(consistency['cosine'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.9]))
    consistency['cosine'].hist(bins=50)
    plt.title('Cosmic target/ref face embedding cosine')

In [ ]:
def simulate_cosmic_filters(df):
    base = df.copy()
    filters = []
    thresholds = [64, 96, 128, 160, 192, 224, 256, 320]
    for min_side in thresholds:
        mask = base['kept_by_current_class'] & (base['face_min_side'] >= min_side)
        filters.append({
            'filter': f'face_min_side >= {min_side}',
            'kept': int(mask.sum()),
            'kept_pct': float(mask.mean()),
            'median_face_min_side': base.loc[mask, 'face_min_side'].median(),
            'median_face_area_pct': base.loc[mask, 'face_area_pct'].median(),
            'median_prompt_words': base.loc[mask, 'prompt_words'].median(),
        })
    for area_pct in [0.01, 0.02, 0.03, 0.04, 0.06, 0.08]:
        mask = base['kept_by_current_class'] & (base['face_area_pct'] >= area_pct)
        filters.append({
            'filter': f'face_area_pct >= {area_pct:.2f}',
            'kept': int(mask.sum()),
            'kept_pct': float(mask.mean()),
            'median_face_min_side': base.loc[mask, 'face_min_side'].median(),
            'median_face_area_pct': base.loc[mask, 'face_area_pct'].median(),
            'median_prompt_words': base.loc[mask, 'prompt_words'].median(),
        })
    combined_rules = [
        ('min_side>=160 & area>=0.02', (base.face_min_side >= 160) & (base.face_area_pct >= 0.02)),
        ('min_side>=192 & area>=0.03', (base.face_min_side >= 192) & (base.face_area_pct >= 0.03)),
        ('min_side>=224 & area>=0.04', (base.face_min_side >= 224) & (base.face_area_pct >= 0.04)),
        ('min_side>=192 & not_simple_bg', (base.face_min_side >= 192) & (~base.is_simple_background)),
        ('min_side>=192 & refs>=3', (base.face_min_side >= 192) & (base.num_face_paths >= 3)),
    ]
    for name, rule in combined_rules:
        mask = base['kept_by_current_class'] & rule
        filters.append({
            'filter': name,
            'kept': int(mask.sum()),
            'kept_pct': float(mask.mean()),
            'median_face_min_side': base.loc[mask, 'face_min_side'].median(),
            'median_face_area_pct': base.loc[mask, 'face_area_pct'].median(),
            'median_prompt_words': base.loc[mask, 'prompt_words'].median(),
        })
    return pd.DataFrame(filters).sort_values(['kept_pct', 'filter'], ascending=[False, True])

filter_table = simulate_cosmic_filters(cosmic_df)
display(filter_table)

# Compare a candidate strict cosmic subset to large_dataset_serv.
candidate_mask = cosmic_df['kept_by_current_class'] & (cosmic_df.face_min_side >= 192) & (cosmic_df.face_area_pct >= 0.03)
candidate_cosmic = cosmic_df[candidate_mask]
print('Candidate cosmic subset rows:', len(candidate_cosmic))
display(pd.concat([
    large_df.assign(dataset='large_dataset_serv'),
    candidate_cosmic.assign(dataset='cosmic_candidate_filtered'),
]).groupby('dataset').agg(
    n=('dataset', 'size'),
    median_face_min_side=('face_min_side', 'median'),
    median_face_area_pct=('face_area_pct', 'median'),
    p10_face_min_side=('face_min_side', lambda s: s.quantile(0.10)),
    median_prompt_words=('prompt_words', 'median'),
))

show_examples(candidate_cosmic, 'cosmic_large', n=8, order='random')

## Interpretation Checklist

Use the outputs above to decide whether `cosmic_large` is worse because of data quality or because of training settings.

Key checks:

- If `cosmic_large` has much smaller `face_min_side` / `face_area_pct`, masked face loss supervises fewer latent pixels and the target identity is harder to reconstruct.
- If many random visual examples show tiny faces, multiple people, wrong target/ref identity, blur, watermarks, crops, or heavy occlusion, the larger dataset is noisier rather than better.
- If the optional InsightFace cosine is low or has many detection failures, filter or repair `face_paths` / target-ref pairing.
- Compare `cosmic_large` with `num_refs=1` and matched effective batch size before judging against `large_dataset_serv`; the launch scripts currently differ in batch size.

Practical fixes to test:

- Filter cosmic by `face_min_side >= 192` or `>= 224`, and possibly `face_area_pct >= 0.03` or `>= 0.04`.
- Remove samples with bad target/ref face-embedding consistency if InsightFace is available.
- Keep `num_refs=1` for the first controlled comparison, then reintroduce multi-ref only after quality improves.
- Consider a curriculum: train first on the stricter cosmic subset, then mix in the larger/noisier remainder at lower weight.

## Additional Diagnostics Added After Server Run

These cells quantify two training-specific issues that are easy to miss from aggregate dataset size: how many target faces are small in latent space, and how different the reference image distribution is from the target image distribution.


In [ ]:
def training_relevant_face_stats(df, name):
    d = df.copy()
    if 'kept_by_current_class' in d.columns:
        d = d[d['kept_by_current_class']]
    d['latent_face_min_side'] = d['face_min_side'] / 8.0
    d['latent_face_area_px'] = d['face_area'] / 64.0
    return {
        'dataset': name,
        'n': len(d),
        'face_min_median_px': d['face_min_side'].median(),
        'latent_min_median_px': d['latent_face_min_side'].median(),
        'latent_area_median_px': d['latent_face_area_px'].median(),
        'pct_face_min_lt_128': (d['face_min_side'] < 128).mean(),
        'pct_face_min_lt_160': (d['face_min_side'] < 160).mean(),
        'pct_face_min_lt_192': (d['face_min_side'] < 192).mean(),
        'pct_face_area_lt_002': (d['face_area_pct'] < 0.02).mean(),
        'pct_face_area_lt_004': (d['face_area_pct'] < 0.04).mean(),
    }

face_training_stats = pd.DataFrame([
    training_relevant_face_stats(large_df, 'large_dataset_serv'),
    training_relevant_face_stats(cosmic_df, 'cosmic_large_current'),
])
display(face_training_stats)

fig, ax = plt.subplots(figsize=(8, 4))
for label, df in [('large', large_df), ('cosmic', cosmic_df[cosmic_df.kept_by_current_class])]:
    vals = np.sort(df['face_min_side'].dropna().to_numpy())
    y = np.linspace(0, 1, len(vals), endpoint=True)
    ax.plot(vals, y, label=label)
for x in [128, 160, 192, 224, 256]:
    ax.axvline(x, color='gray', alpha=0.25, linewidth=1)
    ax.text(x + 2, 0.03, str(x), rotation=90, color='gray')
ax.set_xlabel('target face min side, pixels')
ax.set_ylabel('cumulative fraction')
ax.set_title('Target face size CDF: small faces dominate cosmic_large')
ax.legend()
plt.tight_layout()


In [ ]:
# Grouped target image quality summary. The earlier quality table is combined across datasets.
if 'quality' in globals():
    target_quality_grouped = quality[quality.opened_ok].groupby('dataset').agg(
        n=('dataset', 'size'),
        median_brightness=('brightness_mean', 'median'),
        p05_brightness=('brightness_mean', lambda s: s.quantile(0.05)),
        p95_brightness=('brightness_mean', lambda s: s.quantile(0.95)),
        median_contrast=('brightness_std', 'median'),
        median_sharpness=('sharpness_grad_mean', 'median'),
        p10_sharpness=('sharpness_grad_mean', lambda s: s.quantile(0.10)),
        median_saturation=('saturation_mean', 'median'),
        median_dark_pixel_rate=('dark_pixel_rate', 'median'),
        median_bright_pixel_rate=('bright_pixel_rate', 'median'),
    )
    display(target_quality_grouped)
else:
    print('Run the image-quality sampling cell first; variable `quality` is not defined.')


In [ ]:
def actual_bbox_metrics(box, img_size):
    w, h = img_size
    if box is None:
        return {'valid_ref_bbox': False}
    x0, y0, x1, y1 = [float(v) for v in box]
    bw, bh = x1 - x0, y1 - y0
    area = max(0, bw) * max(0, bh)
    return {
        'valid_ref_bbox': bw > 0 and bh > 0,
        'ref_img_w': w,
        'ref_img_h': h,
        'ref_face_min_side': min(bw, bh),
        'ref_face_area_pct_of_ref': area / float(w * h),
    }

def collect_reference_distribution_stats(n=800):
    rows = []
    large_sample = large_df.sample(min(n, len(large_df)), random_state=RANDOM_SEED)
    for _, row in large_sample.iterrows():
        try:
            ref_img, ref_bbox = get_large_ref(row)
            if ref_img is None:
                continue
            q = image_quality_metrics(ref_img)
            q.update(actual_bbox_metrics(ref_bbox, ref_img.size))
            q.update({'dataset': 'large_dataset_serv_ref', 'target_face_area_pct': row.face_area_pct})
            rows.append(q)
        except Exception as e:
            rows.append({'dataset': 'large_dataset_serv_ref', 'error': repr(e)})

    cosmic_sample = cosmic_df[cosmic_df.kept_by_current_class].sample(min(n, int(cosmic_df.kept_by_current_class.sum())), random_state=RANDOM_SEED)
    for _, row in cosmic_sample.iterrows():
        try:
            ref_img, ref_bbox = get_cosmic_ref(row)
            q = image_quality_metrics(ref_img)
            q.update(actual_bbox_metrics(ref_bbox, ref_img.size))
            q.update({'dataset': 'cosmic_large_ref_after_dataset_crop', 'target_face_area_pct': row.face_area_pct})
            rows.append(q)
        except Exception as e:
            rows.append({'dataset': 'cosmic_large_ref_after_dataset_crop', 'error': repr(e)})
    out = pd.DataFrame(rows)
    out['target_to_ref_face_area_ratio'] = out['target_face_area_pct'] / out['ref_face_area_pct_of_ref']
    return out

ref_distribution = collect_reference_distribution_stats(n=800)
display(ref_distribution.groupby('dataset').agg(
    n=('dataset', 'size'),
    error_rate=('error', lambda s: s.notna().mean() if 'error' in ref_distribution.columns else 0),
    median_ref_w=('ref_img_w', 'median'),
    median_ref_h=('ref_img_h', 'median'),
    median_ref_face_area_pct=('ref_face_area_pct_of_ref', 'median'),
    median_target_face_area_pct=('target_face_area_pct', 'median'),
    median_target_to_ref_face_area_ratio=('target_to_ref_face_area_ratio', 'median'),
    median_ref_sharpness=('sharpness_grad_mean', 'median'),
    median_ref_brightness=('brightness_mean', 'median'),
))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, grp in ref_distribution.dropna(subset=['ref_face_area_pct_of_ref']).groupby('dataset'):
    axes[0].hist(grp['ref_face_area_pct_of_ref'], bins=50, alpha=0.45, density=True, label=label)
    axes[1].hist(grp['target_to_ref_face_area_ratio'].clip(0, 1), bins=50, alpha=0.45, density=True, label=label)
axes[0].set_title('How much of reference image is face?')
axes[0].set_xlabel('ref face area / ref image area')
axes[1].set_title('Target face area vs reference face area')
axes[1].set_xlabel('target face area fraction / ref face area fraction')
for ax in axes:
    ax.legend()
plt.tight_layout()


In [ ]:
# Cheap prompt/source heuristics. These do not prove bad data, but they quickly surface subsets worth visual inspection.
BAD_OR_RISKY_TERMS = [
    'poster', 'logo', 'text', 'sign', 'screen', 'phone', 'selfie', 'advertisement', 'ad ',
    'watermark', 'illustration', 'drawing', 'cartoon', 'anime', 'painting', 'render',
    'baby', 'child', 'kid', 'wedding', 'bride', 'group', 'crowd', 'multiple',
]

def prompt_term_rates(df, name):
    text = df['prompt'].fillna('').str.lower()
    row = {'dataset': name, 'n': len(df)}
    for term in BAD_OR_RISKY_TERMS:
        safe_term = term.strip().replace(" ", "_")
        row[f"prompt_has_{safe_term}"] = text.str.contains(term, regex=False).mean()
    return row

prompt_risk = pd.DataFrame([
    prompt_term_rates(large_df, 'large_dataset_serv'),
    prompt_term_rates(cosmic_df[cosmic_df.kept_by_current_class], 'cosmic_large_current'),
]).set_index('dataset')
display(prompt_risk.T.sort_values('cosmic_large_current', ascending=False).head(30))

# Print random examples from high-risk cosmic prompt subsets for manual review.
risk_mask = cosmic_df.kept_by_current_class & cosmic_df['prompt'].str.lower().str.contains('|'.join(['poster', 'logo', 'screen', 'phone', 'illustration', 'drawing', 'cartoon', 'advertisement', 'watermark']), regex=True, na=False)
print('Cosmic high-risk prompt subset:', int(risk_mask.sum()), 'of', int(cosmic_df.kept_by_current_class.sum()))
display(cosmic_df.loc[risk_mask, ['image_path', 'face_min_side', 'face_area_pct', 'prompt']].sample(min(20, int(risk_mask.sum())), random_state=RANDOM_SEED))


In [ ]:
# Optional heavier identity-consistency check for both datasets. Requires insightface.
# Turn this on when running on a machine with the same face-analysis dependencies as training.
RUN_INSIGHTFACE_CONSISTENCY_BOTH = False
BOTH_CONSISTENCY_SAMPLE = 300

if RUN_INSIGHTFACE_CONSISTENCY_BOTH:
    from insightface.app import FaceAnalysis
    face_app = FaceAnalysis(name='buffalo_l')
    face_app.prepare(ctx_id=0, det_size=(640, 640))

    def largest_face_embedding(img):
        arr = np.asarray(img.convert('RGB'))[:, :, ::-1]
        faces = face_app.get(arr)
        if not faces:
            return None
        faces = sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]), reverse=True)
        emb = faces[0].embedding.astype(np.float32)
        return emb / max(np.linalg.norm(emb), 1e-8)

    def crop_target_face_for_embedding(dataset_name, row):
        if dataset_name == 'large_dataset_serv':
            img = load_large_target(row)
            meta = large_raw[row.identity][row.image_id]
            box = meta['new_face_crop']
        else:
            meta = cosmic_raw[row['image_path']]
            img = load_cosmic_target(row, meta)
            box = meta['face_crop_new']
        x0, y0, x1, y1 = [int(v) for v in box]
        return img.crop((x0, y0, x1, y1))

    rows = []
    for dataset_name, df in [('large_dataset_serv', large_df), ('cosmic_large', cosmic_df[cosmic_df.kept_by_current_class])]:
        sample = df.sample(min(BOTH_CONSISTENCY_SAMPLE, len(df)), random_state=RANDOM_SEED)
        for _, row in sample.iterrows():
            try:
                tgt_face = crop_target_face_for_embedding(dataset_name, row)
                ref_img, ref_bbox = get_large_ref(row) if dataset_name == 'large_dataset_serv' else get_cosmic_ref(row)
                x0, y0, x1, y1 = [int(v) for v in ref_bbox]
                ref_face = ref_img.crop((x0, y0, x1, y1))
                e_tgt = largest_face_embedding(tgt_face)
                e_ref = largest_face_embedding(ref_face)
                cosine = np.nan if e_tgt is None or e_ref is None else float(np.dot(e_tgt, e_ref))
                rows.append({'dataset': dataset_name, 'cosine': cosine, 'ok': np.isfinite(cosine)})
            except Exception as e:
                rows.append({'dataset': dataset_name, 'cosine': np.nan, 'ok': False, 'error': repr(e)})
    consistency_both = pd.DataFrame(rows)
    display(consistency_both.groupby('dataset').agg(
        ok_rate=('ok', 'mean'),
        median_cosine=('cosine', 'median'),
        p10_cosine=('cosine', lambda s: s.quantile(0.10)),
        p25_cosine=('cosine', lambda s: s.quantile(0.25)),
    ))
    consistency_both.dropna(subset=['cosine']).hist(column='cosine', by='dataset', bins=50, figsize=(10, 4), sharex=True)


## Four-Way Dataset / Filter Comparison

`large_dataset_serv` has explicit cross-image identities, where samples are target images. For `cosmic_large`, ID = target image and samples = total number of images in `face_paths` for the selected targets; therefore `images_per_id` is `num_face_paths` per target.

In [ ]:
def summarize_training_option(name, df, *, identity_col=None, sample_count_col=None, identity_definition='explicit identity'):
    d = df.copy()
    if identity_col is not None and identity_col in d.columns:
        image_counts = d.groupby(identity_col).size()
        num_ids = int(image_counts.shape[0])
        samples = int(len(d))
    elif sample_count_col is not None and sample_count_col in d.columns:
        image_counts = pd.to_numeric(d[sample_count_col], errors='coerce').fillna(0).astype(np.int64)
        num_ids = int(len(d))
        samples = int(image_counts.sum())
    else:
        image_counts = pd.Series(np.ones(len(d), dtype=np.int64))
        num_ids = int(len(d))
        samples = int(len(d))

    row = {
        'option': name,
        'samples': samples,
        'id_definition': identity_definition,
        'num_ids': num_ids,
        'images_per_id_mean': float(image_counts.mean()) if len(image_counts) else np.nan,
        'images_per_id_median': float(image_counts.median()) if len(image_counts) else np.nan,
        'images_per_id_p90': float(image_counts.quantile(0.90)) if len(image_counts) else np.nan,
        'images_per_id_max': int(image_counts.max()) if len(image_counts) else 0,
        'face_min_p05': float(d['face_min_side'].quantile(0.05)),
        'face_min_p10': float(d['face_min_side'].quantile(0.10)),
        'face_min_p25': float(d['face_min_side'].quantile(0.25)),
        'face_min_median': float(d['face_min_side'].median()),
        'face_min_p75': float(d['face_min_side'].quantile(0.75)),
        'face_min_p90': float(d['face_min_side'].quantile(0.90)),
        'face_area_pct_p10': float(d['face_area_pct'].quantile(0.10)),
        'face_area_pct_p25': float(d['face_area_pct'].quantile(0.25)),
        'face_area_pct_median': float(d['face_area_pct'].median()),
        'face_area_pct_p75': float(d['face_area_pct'].quantile(0.75)),
        'face_area_pct_p90': float(d['face_area_pct'].quantile(0.90)),
        'pct_face_min_lt_128': float((d['face_min_side'] < 128).mean()),
        'pct_face_min_lt_160': float((d['face_min_side'] < 160).mean()),
        'pct_face_min_lt_192': float((d['face_min_side'] < 192).mean()),
    }

    if 'num_face_paths' in d.columns:
        refs = d['num_face_paths']
        row.update({
            'ref_images_per_target_median': float(refs.median()),
            'ref_images_per_target_p10': float(refs.quantile(0.10)),
            'ref_images_per_target_p90': float(refs.quantile(0.90)),
        })
    elif 'num_identity_images' in d.columns:
        refs = (d['num_identity_images'] - 1).clip(lower=0)
        row.update({
            'same_id_ref_candidates_median': float(refs.median()),
            'same_id_ref_candidates_p10': float(refs.quantile(0.10)),
            'same_id_ref_candidates_p90': float(refs.quantile(0.90)),
        })
    return row

cosmic_current = cosmic_df[cosmic_df['kept_by_current_class']].copy()
cosmic_min192 = cosmic_current[cosmic_current['face_min_side'] >= 192].copy()
cosmic_min256 = cosmic_current[cosmic_current['face_min_side'] >= 256].copy()

four_way_compare = pd.DataFrame([
    summarize_training_option(
        'large_dataset_serv (current)',
        large_df,
        identity_col='identity',
        identity_definition='explicit identity from JSON',
    ),
    summarize_training_option(
        'cosmic_large (current min_face_res=64)',
        cosmic_current,
        identity_col=None,
        sample_count_col='num_face_paths',
        identity_definition='target image; samples=sum(face_paths)',
    ),
    summarize_training_option(
        'cosmic_large (min_face_res=192)',
        cosmic_min192,
        identity_col=None,
        sample_count_col='num_face_paths',
        identity_definition='target image; samples=sum(face_paths)',
    ),
    summarize_training_option(
        'cosmic_large (min_face_res=256)',
        cosmic_min256,
        identity_col=None,
        sample_count_col='num_face_paths',
        identity_definition='target image; samples=sum(face_paths)',
    ),
])

display(four_way_compare)

face_distribution_cols = [
    'option', 'samples', 'num_ids', 'images_per_id_mean', 'images_per_id_median', 'images_per_id_p90',
    'face_min_p10', 'face_min_p25', 'face_min_median', 'face_min_p75', 'face_min_p90',
    'face_area_pct_p10', 'face_area_pct_p25', 'face_area_pct_median', 'face_area_pct_p75', 'face_area_pct_p90',
    'pct_face_min_lt_128', 'pct_face_min_lt_160', 'pct_face_min_lt_192',
]
display(four_way_compare[face_distribution_cols])
